In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy, pandas
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [3]:
os.makedirs('data', exist_ok=True)
os.makedirs('model', exist_ok=True)

In [4]:
train = pandas.read_csv('/kaggle/input/zelestra-phase-2-data/Dataset 1.csv')

In [5]:
train = train[['meteorolgicas_em_08_01_gii', 'celulas_ctin08_cc_08_2_ir_cel_2', 'celulas_ctin08_cc_08_2_ir_cel_1', 
               'celulas_ctin08_cc_08_1_ir_cel_2', 'celulas_ctin08_cc_08_1_ir_cel_1', 'meteorolgicas_em_08_01_ghi', 
               'meteorolgicas_em_03_02_gii', 'celulas_ctin03_cc_03_1_ir_cel_1', 'celulas_ctin03_cc_03_2_ir_cel_2', 
               'celulas_ctin03_cc_03_1_ir_cel_2', 'celulas_ctin03_cc_03_2_ir_cel_1', 'meteorolgicas_em_03_02_ghi', 
               'celulas_ctin08_cc_08_1_t_mod', 'celulas_ctin08_cc_08_2_t_mod', 'meteorolgicas_em_08_01_gii_rear', 
               'inversores_ctin08_inv_08_08_p', 'inversores_ctin08_inv_08_08_p_dc', 'inversores_ctin08_inv_08_08_eact_tot', 
               'meteorolgicas_em_03_02_gii_rear', 'celulas_ctin03_cc_03_1_t_mod', 'celulas_ctin03_cc_03_2_t_mod', 
               'inversores_ctin03_inv_03_03_p_dc', 'inversores_ctin03_inv_03_03_p', 'inversores_ctin03_inv_03_03_eact_tot', 
               'celulas_ctin08_cc_08_2_t_amb', 'celulas_ctin08_cc_08_1_t_amb', 'celulas_ctin03_cc_03_2_t_amb', 
               'celulas_ctin03_cc_03_1_t_amb', 'inversores_ctin03_strings_string10_pv_i9', 'ttr_potenciaproducible']]

In [6]:
train, test = train_test_split(train, test_size=0.1, random_state=0)

In [7]:
train.to_csv('data/train_r.csv', index = False)
test.to_csv('data/test_r.csv', index = False)

In [8]:
test

,meteorolgicas_em_08_01_gii,celulas_ctin08_cc_08_2_ir_cel_2,celulas_ctin08_cc_08_2_ir_cel_1,celulas_ctin08_cc_08_1_ir_cel_2,celulas_ctin08_cc_08_1_ir_cel_1,meteorolgicas_em_08_01_ghi,meteorolgicas_em_03_02_gii,celulas_ctin03_cc_03_1_ir_cel_1,celulas_ctin03_cc_03_2_ir_cel_2,celulas_ctin03_cc_03_1_ir_cel_2,...,celulas_ctin03_cc_03_2_t_mod,inversores_ctin03_inv_03_03_p_dc,inversores_ctin03_inv_03_03_p,inversores_ctin03_inv_03_03_eact_tot,celulas_ctin08_cc_08_2_t_amb,celulas_ctin08_cc_08_1_t_amb,celulas_ctin03_cc_03_2_t_amb,celulas_ctin03_cc_03_1_t_amb,inversores_ctin03_strings_string10_pv_i9,ttr_potenciaproducible
16967,104.947368,98.937500,97.235294,100.391304,100.000000,94.300000,109.410256,100.857143,104.444444,104.428571,...,22.860000,0.459806,0.453000,0.112,18.857143,18.600000,20.211111,19.687500,3.042880,5.214632
14256,794.977011,799.055556,796.000000,796.111111,800.809524,711.344828,807.091954,821.600000,820.700000,820.846154,...,49.357143,3.558200,3.486811,0.879,35.441667,28.507143,35.466667,31.980000,21.354780,38.710974
1310,645.141176,635.500000,646.444444,647.653846,647.913043,553.880435,663.069767,671.571429,632.454545,662.000000,...,50.100000,2.781811,2.728233,0.690,36.363636,35.015384,39.175000,33.945454,16.434513,33.368090
5720,375.977528,374.750000,364.521739,376.529412,383.961538,356.391304,443.775281,454.812500,444.400000,459.642857,...,38.855556,1.880505,1.849132,0.460,32.918183,29.940000,34.000000,29.100000,10.661463,21.344444
14912,5.272727,4.750000,3.800000,4.750000,3.222222,4.100000,4.833333,3.666667,3.666667,4.000000,...,13.300000,0.000000,0.000000,0.000,13.342857,13.300000,13.300000,13.100000,0.141000,0.039860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10691,101.571429,94.368421,89.000000,86.040000,96.291667,54.733333,94.631579,94.466667,95.416667,99.933333,...,2.887500,0.391415,0.384031,0.090,2.054545,1.463636,4.512500,3.300000,2.488270,5.066593
9656,116.000000,115.933333,113.466667,114.500000,114.235294,124.000000,126.571429,119.272727,118.666667,122.666667,...,23.420000,0.541379,0.530745,0.135,21.800000,21.200000,21.250000,21.000000,3.260129,6.288987
7967,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,11.566666,0.000000,0.000000,0.000,10.771429,10.533333,13.171429,12.254546,NaN,0.000000
7292,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.500000,0.000000,0.000000,0.000,3.525000,3.785714,6.875000,6.233333,NaN,0.000000


In [9]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

for i in zip(train.columns, train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]] = normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].abs().mean(), inplace=True)
        train[i[0]] = normalize.fit_transform(numpy.array(train[i[0]].abs()).reshape(-1,1))
for i in zip(test.columns, test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]] = normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].abs().mean(), inplace=True)
        test[i[0]] = normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [10]:
train_c_x, train_c_y = train.drop(columns = ['ttr_potenciaproducible']), train['ttr_potenciaproducible']

train_c_x = train_c_x.reset_index().drop(columns=['index'])
train_c_y = train_c_y.reset_index().drop(columns=['index'])

test_c_x, test_c_y = test.drop(columns = ['ttr_potenciaproducible']), test['ttr_potenciaproducible']

test_c_x = test_c_x.reset_index().drop(columns=['index'])
test_c_y = test_c_y.reset_index().drop(columns=['index'])

In [11]:
train_c_x

,meteorolgicas_em_08_01_gii,celulas_ctin08_cc_08_2_ir_cel_2,celulas_ctin08_cc_08_2_ir_cel_1,celulas_ctin08_cc_08_1_ir_cel_2,celulas_ctin08_cc_08_1_ir_cel_1,meteorolgicas_em_08_01_ghi,meteorolgicas_em_03_02_gii,celulas_ctin03_cc_03_1_ir_cel_1,celulas_ctin03_cc_03_2_ir_cel_2,celulas_ctin03_cc_03_1_ir_cel_2,...,celulas_ctin03_cc_03_1_t_mod,celulas_ctin03_cc_03_2_t_mod,inversores_ctin03_inv_03_03_p_dc,inversores_ctin03_inv_03_03_p,inversores_ctin03_inv_03_03_eact_tot,celulas_ctin08_cc_08_2_t_amb,celulas_ctin08_cc_08_1_t_amb,celulas_ctin03_cc_03_2_t_amb,celulas_ctin03_cc_03_1_t_amb,inversores_ctin03_strings_string10_pv_i9
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.016667,7.266667,0.000000,0.000000,0.000,7.125000,6.871429,7.600000,7.333333,7.482453
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.920000,5.120000,0.000000,0.000000,0.000,4.750000,4.812500,6.200000,5.650000,7.482453
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10.412500,10.516667,0.000000,0.000000,0.000,11.458333,11.550000,11.383333,11.133333,7.482453
3,1.000000,0.000000,0.000000,1.666667,0.000000,1.000000,1.500000,0.000000,2.500000,2.500000,...,18.850000,19.262500,0.000000,0.000000,0.001,20.075000,20.500000,19.980000,19.550000,0.000000
4,92.914634,91.263158,88.800000,85.148148,89.000000,70.550000,122.208955,118.750000,120.272727,115.500000,...,21.072727,22.270000,0.439574,0.432011,0.108,18.066667,16.566667,19.144444,16.325000,2.658323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15719,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.200000,5.540000,0.000000,0.000000,0.000,6.840000,7.066667,7.050000,6.866667,7.482453
15720,422.923077,424.631579,421.619048,431.434783,441.450000,282.043956,440.526882,449.437500,439.363636,446.250000,...,40.155555,38.966666,1.726511,1.689833,0.425,28.607143,25.420000,31.500000,29.110000,9.317000
15721,399.111111,378.052632,365.772727,372.296296,376.583333,545.208333,512.274510,498.153846,490.100000,499.416667,...,33.357143,34.020000,2.119385,2.081674,0.526,32.433334,29.188889,26.871429,26.137500,12.160447
15722,202.062500,209.615385,208.909091,209.000000,209.285714,205.500000,204.363636,206.250000,208.000000,205.700000,...,23.471428,23.762500,0.977800,0.958105,0.241,19.940000,18.985714,18.420000,18.590000,6.060645


In [12]:
class solo_models:
    def __new__(self, train_c_x, train_c_y, test_c_x, test_c_y):
        print('INITIATING PARAMETER')
        self.X_train, self.X_test, self.y_train, self.y_test = train_c_x, test_c_x, train_c_y, test_c_y
        #print("train_c_x:",self.train_c_x.index,"train_c_y:",self.train_c_y.index,"X_tain:",self.X_train.index)
        self.R_Forest_parm = {'n_estimators' : 25, 
                              'min_samples_split' : 2, 
                              'max_depth' : 10, 
                              'min_samples_leaf' : 2, 
                              'random_state' : 42}
        
        self.Extra_parm = {'n_estimators' : 50, 
                           'min_samples_split' : 2, 
                           'max_depth' : 8, 
                           'min_samples_leaf' : 2, 
                           'random_state' : 42}
        
        self.LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
                           'drop_rate': 0.8303473371870002,
                           'learning_rate': 0.2762739125344964,
                           'max_bin': 9983,
                           'max_depth': 8623,
                           'max_drop': 5480,
                           'min_child_samples': 101,
                           'min_data_in_leaf': 426,
                           'n_estimators': 1628,
                           'num_leaves': 3640,
                           'objective': 'regression_l1',
                           'reg_alpha': 0.39940189926691194,
                           'reg_lambda': 0.9567353299218986,
                           'skip_drop': 0.6274640597528743,
                           'verbosity': -1}
        
        self.XGB_R_parm = {'colsample_bytree' : 0.871893537724492,
                           'gamma' : 1,
                           'learning_rate' : 0.923192518624813,
                           'max_depth' : 15,
                           'min_child_weight' : 1,
                           'n_estimators' : 17748,
                           'reg_alpha' : 45,
                           'reg_lambda' : 0.8598696247943665,
                           'subsample' : 0.9109367356405415,
                           'random_state' : 42}

        self.catboost_params = {'iterations' : 3000,
                                'learning_rate': 0.009, 
                                'depth': 5, 
                                'l2_leaf_reg': 5.5,
                                'min_child_samples' : 102,
                                'od_wait' : 50,
                                'random_state' : 42,
                                'eval_metric': 'RMSE', 
                                'od_type' : 'Iter',
                                'bootstrap_type': 'Bayesian', 
                                'grow_policy' : 'Depthwise',
                                'logging_level' : 'Silent'}
        
        self.DecisionTree = {'max_depth': 6}
        self.settings = {"time_budget": 2000,
                         "metric": "mae",
                         "estimator_list": ["lgbm"],
                         "task": "regression",
                         "log_file_name": "experiment.log",
                         "seed": 41,
                         "eval_method": "cv"
                    }
        self.X_train.to_csv('data/X_train.csv', index = False)
        self.y_train.to_csv('data/y_train.csv', index = False)
        self.X_test.to_csv('data/X_test.csv', index = False)
        self.y_test.to_csv('data/y_test.csv', index = False)
        
        self.models(self)
        self.manual_training(self)
        return {'manual_ensemble' : self.model_collecter, 'stack_ensemble' : self.stack_training(self)}
        
    def models(self):
        print('LOADING MODEL')
        self.model_collecter = {}
        
        self.LGBM_R = LGBMRegressor(**self.LGBM_R_parm)
        self.XGB_R = XGBRegressor(**self.XGB_R_parm)
        self.catboost = CatBoostRegressor(**self.catboost_params)

        self.model_collecter['random_forest'] = BaggingRegressor(estimator = RandomForestRegressor(**self.R_Forest_parm), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['extra_trees'] = BaggingRegressor(estimator = ExtraTreesRegressor(**self.Extra_parm), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['GradientBoostingRegressor'] = BaggingRegressor(estimator = GradientBoostingRegressor(learning_rate=0.1, 
                                                                                                                   min_samples_split=500,
                                                                                                                   min_samples_leaf=50,
                                                                                                                   max_depth=8,
                                                                                                                   max_features='sqrt',
                                                                                                                   subsample=0.8,
                                                                                                                   random_state=10), 
                                                                            n_estimators = 10, random_state = 0)
        
        self.model_collecter['DecisionTreeRegressor'] = BaggingRegressor(estimator = DecisionTreeRegressor(**self.DecisionTree), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['TweedieRegressor'] = BaggingRegressor(estimator = TweedieRegressor(), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['LinearRegression'] = BaggingRegressor(estimator = LinearRegression(), 
                                                                 n_estimators = 10, random_state = 0)
        
        self.model_collecter['ElasticNet'] = BaggingRegressor(estimator = ElasticNetCV(), n_estimators = 10, random_state = 0)
        self.model_collecter['LassoCV'] = BaggingRegressor(estimator = LassoCV(), n_estimators = 10, random_state = 0)
        self.model_collecter['RidgeCV'] = BaggingRegressor(estimator = RidgeCV(), n_estimators = 10, random_state = 0)
        self.model_collecter['LassoLars'] = BaggingRegressor(estimator = LassoLars(), n_estimators = 10, random_state = 0)
        self.model_collecter['Lasso'] = BaggingRegressor(estimator = Lasso(), n_estimators = 10, random_state = 0)
        self.model_collecter['BayesianRidge'] = BaggingRegressor(estimator = BayesianRidge(), n_estimators = 10, random_state = 0)
        self.model_collecter['SVR'] = BaggingRegressor(estimator = SVR(), n_estimators = 10, random_state = 0)
        self.model_collecter['PLSRegression'] = BaggingRegressor(estimator = PLSRegression(), n_estimators = 10, random_state = 0)
        self.model_collecter['KMeans'] = BaggingRegressor(estimator = KMeans(), n_estimators = 10, random_state = 0)
        #self.model_collecter['GaussianProcessRegressor'] = GaussianProcessRegressor()
        self.model_collecter['MLPRegressor'] = BaggingRegressor(estimator = MLPRegressor(), n_estimators = 10, random_state = 0)
        
    def manual_training(self):
        print('TRAINING')
        
        for i in self.model_collecter.keys():
            print(f'--{i.upper()}')
            #print(self.X_train)
            self.model_collecter[i].fit(self.X_train, self.y_train)
            joblib.dump(self.model_collecter[i], f'model/{i}.pkl')

        print('--CATBOOST')
        cat_features = list(self.X_train.select_dtypes(include=['object', 'category']).columns)
        train_pool = Pool(self.X_train, self.y_train, cat_features=cat_features)
        val_pool = Pool(self.X_test, self.y_test, cat_features=cat_features)
        
        self.catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)
        print('--XGBOOST')
        self.XGB_R.fit(self.X_train, self.y_train, eval_set = [(self.X_test, self.y_test)],eval_metric = "rmse", verbose = False)
        print('--LGBOOST')
        self.LGBM_R.fit(self.X_train, self.y_train,eval_set = [(self.X_test, self.y_test)],eval_metric ='rmse')
        
        self.model_collecter['LGBMRegressor'] = self.LGBM_R
        self.model_collecter['XGBRegressor'] = self.XGB_R
        self.model_collecter['CatBoostRegressor'] = self.catboost

        joblib.dump(self.model_collecter['LGBMRegressor'], f'model/LGBMRegressor.pkl')
        joblib.dump(self.model_collecter['XGBRegressor'], f'model/XGBRegressor.pkl')
        joblib.dump(self.model_collecter['CatBoostRegressor'], f'model/CatBoostRegressor.pkl')
        
    def stack_training(self):    
        print('--STACK')
        #self.model_collecter['LGBMRegressor'] = self.LGBM_R
        #self.model_collecter['XGBRegressor'] = self.XGB_R
        #self.model_collecter['CatBoostRegressor'] = self.catboost
        self.stack_model = ['LGBMRegressor',
                            'LGBMRegressor',
                            'CatBoostRegressor',
                            'random_forest',
                            'extra_trees',
                            'GradientBoostingRegressor',
                            'DecisionTreeRegressor']
        
        estimators = [(i, self.model_collecter[i]) for i in self.model_collecter.keys() if i in self.stack_model]
        self.model = StackingRegressor(estimators, final_estimator = RidgeCV())
        self.model.fit(self.X_train, self.y_train)
        joblib.dump(self.model, f'model/STACK.pkl')
        return self.model

In [13]:
E_model = solo_models(train_c_x, train_c_y, test_c_x, test_c_y)

INITIATING PARAMETER
LOADING MODEL
TRAINING
--RANDOM_FOREST
--EXTRA_TREES
--GRADIENTBOOSTINGREGRESSOR
--DECISIONTREEREGRESSOR
--TWEEDIEREGRESSOR
--LINEARREGRESSION
--ELASTICNET
--LASSOCV
--RIDGECV
--LASSOLARS
--LASSO
--BAYESIANRIDGE
--SVR
--PLSREGRESSION
--KMEANS
--MLPREGRESSOR
--CATBOOST
--XGBOOST
--LGBOOST
--STACK
